# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

1. **Unit of Analysis:**  
   One row = one unique content item (page) for a given client, aggregated over a specific calendar month

2. **Tables To be Used**
   - `fact_content_daily_performance_sample` (for daily search and analytics signals aggregated to month grain)
   - `dim_content` (for page properties like `word_count` and `content_type` joined via `content_hash_id`)
   - `dim_clients` (for holdout splitting and client profile checks via `client_hash_id`)

3. **Time Window:**  
   A single calendar month: **June 1, 2026 to June 30, 2026** (`month = '2026-06'`), which is the latest complete month partition available in the sample dataset.

4. **What Will be Predicted or Ranked :**  
   We will score and rank pages by their **position-adjusted CTR gap** (defined as `tier_median_ctr - observed_ctr` where the median is computed within the page's position tier). This serves as a ranking proxy for click opportunity. A positive gap tells us a page performs worse than its ranking peers.

5. **Deliberately Excluded:**  
   - `is_declining_label` / `trend_direction` / `trend_pct` for the target month: these are trailing outcomes and would leak target signals.
   - Client IDs and Content IDs as model features: they are pseudonym strings used strictly for joins and client-holdout validation splits.
   - Future GSC/GA4 records: no data from after June 30, 2026 is allowed to enter the features or targets.

In [ ]:
# Connect to Hugging Face and initialize DuckDB
import duckdb
import os, getpass

# CI sets HF_TOKEN in the environment; otherwise prompt
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients': f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
}
print("DuckDB connected and tables configured.")

## 2. Fields: feature / label / context / excluded

| Field Name | Bucket | Role / Rationale | Knowable at Decision Moment? |
| --- | --- | --- | --- |
| `total_impressions` | **Feature** | Sum of GSC impressions | **Yes:** Fully observed over the target month. |
| `avg_position` | **Feature** | Weighted avg position in GSC | **Yes:** Aggregated from recorded queries in GSC. |
| `engagement_rate` | **Feature** | GA4 engaged sessions / total sessions | **Yes:** Analytics sessions are logged before review. |
| `word_count` | **Feature** | Page word count from `dim_content` | **Yes:** Current page length is fully measurable. |
| `position_tier` | **Feature** | Derived bucket from `avg_position` | **Yes:** Computed from observed position records. |
| `ctr_gap` | **Label / Proxy** | `tier_median_ctr - observed_ctr` | **No:** This is our target opportunity metric. |
| `is_opportunity` | **Label / Proxy** | Binary target (`ctr_gap > 0` and `impressions >= 1000`) | **No:** Evaluated post-hoc to calculate Precision@K. |
| `client_hash_id` | **Context** | Grouping and holdout splitting | **Yes:** Used for splitting, not as a model feature. |
| `content_hash_id` | **Context** | Unique identifier for pages | **Yes:** Used for joins, not as a model feature. |
| `observed_ctr` | **Context / Trap** | Raw CTR (`clicks / impressions * 100`) | **No (Excluded):** Directly encodes target outcome. |
| `trend_direction` | **Excluded** | Product trend flags | **No:** Post-hoc trend outcome, would leak information. |
| `trend_pct` | **Excluded** | Raw percentage trend | **No:** Label-derived field, excluded to prevent leakage. |

In [ ]:
# Proof of five features - Build feature frame query
feature_query = f"""
SELECT 
    f.client_hash_id,
    f.content_hash_id,
    SUM(f.gsc_impressions) AS total_impressions,
    CASE WHEN SUM(f.gsc_impressions) > 0 
         THEN CAST(SUM(f.gsc_clicks) AS DOUBLE) / SUM(f.gsc_impressions) * 100.0 
         ELSE 0.0 
    END AS observed_ctr,
    CASE WHEN SUM(f.gsc_impressions) > 0 
         THEN SUM(f.gsc_avg_position * f.gsc_impressions) / SUM(f.gsc_impressions) 
         ELSE 0.0 
    END AS avg_position,
    CASE WHEN SUM(f.ga4_sessions) > 0 
         THEN CAST(SUM(f.ga4_engaged_sessions) AS DOUBLE) / SUM(f.ga4_sessions) * 100.0 
         ELSE 0.0 
    END AS engagement_rate,
    ANY_VALUE(d.word_count) AS word_count
FROM {TABLES['fact_daily_sample']} f
LEFT JOIN {TABLES['dim_content']} d ON f.content_hash_id = d.content_hash_id
WHERE f.month = '2026-06'
GROUP BY f.client_hash_id, f.content_hash_id
HAVING total_impressions >= 500 AND avg_position > 0
LIMIT 5
"""
con.sql(feature_query).df()

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.